In [98]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
os.chdir('E:\PGCP-AI\PML\Cases\Image_Segmentation')

# Image Segmentation

In [100]:
df = pd.read_csv('Image_Segmentation.csv' )
df

,Class,region.centroid.col,region.centroid.row,region.pixel.count,short.line.density.5,short.line.density.2,vedge.mean,vegde.sd,hedge.mean,hedge.sd,intensity.mean,rawred.mean,rawblue.mean,rawgreen.mean,exred.mean,exblue.mean,exgreen.mean,value.mean,saturation.mean,hue-mean
0,BRICKFACE,188,133,9,0.000000,0.0,0.333333,0.266667,0.500000,0.077778,6.666666,8.333334,7.777778,3.888889,5.000000,3.333333,-8.333333,8.444445,0.538580,-0.924817
1,BRICKFACE,105,139,9,0.000000,0.0,0.277778,0.107407,0.833333,0.522222,6.111111,7.555555,7.222222,3.555556,4.333334,3.333333,-7.666666,7.555555,0.532628,-0.965946
2,BRICKFACE,34,137,9,0.000000,0.0,0.500000,0.166667,1.111111,0.474074,5.851852,7.777778,6.444445,3.333333,5.777778,1.777778,-7.555555,7.777778,0.573633,-0.744272
3,BRICKFACE,39,111,9,0.000000,0.0,0.722222,0.374074,0.888889,0.429629,6.037037,7.000000,7.666666,3.444444,2.888889,4.888889,-7.777778,7.888889,0.562919,-1.175773
4,BRICKFACE,16,128,9,0.000000,0.0,0.500000,0.077778,0.666667,0.311111,5.555555,6.888889,6.666666,3.111111,4.000000,3.333333,-7.333334,7.111111,0.561508,-0.985811
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,GRASS,36,243,9,0.111111,0.0,1.888889,1.851851,2.000000,0.711110,13.333333,9.888889,12.111111,18.000000,-10.333333,-3.666667,14.000000,18.000000,0.452229,2.368311
205,GRASS,186,218,9,0.000000,0.0,1.166667,0.744444,1.166667,0.655555,13.703704,10.666667,12.666667,17.777779,-9.111111,-3.111111,12.222222,17.777779,0.401347,2.382684
206,GRASS,197,236,9,0.000000,0.0,2.444444,6.829628,3.333333,7.599998,16.074074,13.111111,16.666668,18.444445,-8.888889,1.777778,7.111111,18.555555,0.292729,2.789800
207,GRASS,208,240,9,0.111111,0.0,1.055556,0.862963,2.444444,5.007407,14.148149,10.888889,13.000000,18.555555,-9.777778,-3.444444,13.222222,18.555555,0.421621,2.392487


In [101]:
X , y = df.drop('Class', axis = 1), df['Class']

In [102]:
y.unique()

array(['BRICKFACE', 'SKY', 'FOLIAGE', 'CEMENT', 'WINDOW', 'PATH', 'GRASS'],
      dtype=object)

In [103]:

clf = IsolationForest(contamination = 0.05 , random_state=26)
clf.fit(X)

predictions = clf.predict(X)
print(predictions)
print('% age of outliers = '+str((predictions<0).mean()*100)+ ' %')
abn_ind = np.where(predictions < 0)
 
abn_ind

[ 1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  1  1  1  1  1  1 -1 -1  1 -1  1  1  1  1  1  1 -1
  1  1  1  1  1  1  1  1  1  1 -1  1  1  1  1  1  1 -1  1  1  1  1  1 -1
  1  1  1  1 -1  1  1  1  1  1 -1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1 -1  1  1  1  1
  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  1  1  1 -1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1]
% age of outliers = 5.263157894736842 %


(array([ 37,  38,  40,  47,  58,  65,  71,  76,  82, 115, 149]),)

In [104]:
pd.crosstab(y,predictions)

col_0,-1,1
Class,,
BRICKFACE,0,29
CEMENT,1,29
FOLIAGE,4,26
GRASS,0,30
PATH,1,29
SKY,5,25
WINDOW,0,30


In [105]:
y[X.index[abn_ind]].value_counts()

Class
SKY        5
FOLIAGE    4
CEMENT     1
PATH       1
Name: count, dtype: int64

In [106]:
df.index[abn_ind]

Index([37, 38, 40, 47, 58, 65, 71, 76, 82, 115, 149], dtype='int64')

# with removing outliers

In [107]:
kfold = StratifiedKFold(n_splits=5,shuffle=True, random_state=26)
lr = LogisticRegression()
lr.get_params()

{'C': 1.0,
 'class_weight': None,
 'dual': False,
 'fit_intercept': True,
 'intercept_scaling': 1,
 'l1_ratio': None,
 'max_iter': 100,
 'multi_class': 'deprecated',
 'n_jobs': None,
 'penalty': 'l2',
 'random_state': None,
 'solver': 'lbfgs',
 'tol': 0.0001,
 'verbose': 0,
 'warm_start': False}

In [108]:
df = df[predictions ==1 ]
X, y = df.drop("Class",axis=1),df["Class"]

In [109]:
params = { "C":np.linspace(0.001,5,10),
         "solver" : ['lbfgs', 'newton-cg','newton-cholesky', 'sag', 'saga']}
gcv = GridSearchCV(lr, param_grid=params, cv = kfold, scoring="neg_log_loss" , n_jobs=-1)
gcv.fit(X,y)

,estimator,LogisticRegression()
,param_grid,"{'C': array([1.0000...00000000e+00]), 'solver': ['lbfgs', 'newton-cg', ...]}"
,scoring,'neg_log_loss'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [110]:
gcv.best_params_ , gcv.best_score_

({'C': np.float64(0.001), 'solver': 'newton-cg'},
 np.float64(-0.47603470864178876))

# without removing outliers

In [116]:
os.chdir('E:\PGCP-AI\PML\Cases\Satellite_Imaging')

In [117]:
df = pd.read_csv('Satellite.csv', sep=';')
X , y  = df.drop('classes', axis=1), df['classes']

In [118]:
lr = LogisticRegression() 
params = { "C":np.linspace(0.001,5,10),
         "solver" : ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga']}
kfold=  StratifiedKFold(n_splits=5,shuffle=True, random_state=26)

In [119]:
gcv = GridSearchCV(lr, param_grid=params, cv = kfold, scoring="neg_log_loss" , n_jobs=-1)
gcv.fit(X,y)

,estimator,LogisticRegression()
,param_grid,"{'C': array([1.0000...00000000e+00]), 'solver': ['lbfgs', 'newton-cg', ...]}"
,scoring,'neg_log_loss'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [120]:
gcv.best_params_ , gcv.best_score_

({'C': np.float64(0.001), 'solver': 'newton-cholesky'},
 np.float64(-0.3596180430217847))

In [123]:
clf = IsolationForest(contamination = 0.05 , random_state=26)
clf.fit(X)

predictions = clf.predict(X)
print(predictions)
print('% age of outliers = '+str((predictions<0).mean()*100)+ ' %')
abn_ind = np.where(predictions < 0)
X.index[abn_ind]

[1 1 1 ... 1 1 1]
% age of outliers = 5.003885003885004 %


Index([ 133,  134,  188,  204,  205,  206,  250,  251,  267,  268,
       ...
       5178, 5180, 5234, 5302, 5543, 5561, 5562, 5658, 5732, 6129],
      dtype='int64', length=322)

In [124]:
df = df[predictions ==1 ]
X, y = df.drop("classes",axis=1),df["classes"]

In [126]:
lr = LogisticRegression() 
params = { "C":np.linspace(0.001,5,10),
         "solver" : ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga']}
kfold=  StratifiedKFold(n_splits=5,shuffle=True, random_state=26)

In [127]:
gcv = GridSearchCV(lr, param_grid=params, cv = kfold, scoring="neg_log_loss" , n_jobs=-1)
gcv.fit(X,y)

,estimator,LogisticRegression()
,param_grid,"{'C': array([1.0000...00000000e+00]), 'solver': ['lbfgs', 'newton-cg', ...]}"
,scoring,'neg_log_loss'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [128]:
gcv.best_params_ , gcv.best_score_

({'C': np.float64(0.001), 'solver': 'newton-cg'},
 np.float64(-0.3679824079629931))